# LLM-as-Judge: Entity Code Behavioral Verification

Two-pass Gemini judge that verifies generated entity code against:
1. Its policy-callable interface (method names, signatures, on_query attrs)
2. Cross-entity interface compatibility via the mediating policy (Pass 2)

**Architecture:** Entities are stateful hook providers. Policies are behaviour adapters.
See `docs/generated_code_evaluation_method.md` for full methodology.

**Kernel**: Use the backend env — `engine/backend/env/bin/python`

## ⚙️ Config — edit this cell

In [165]:
import os

# Path to the exported code-workspace directory
WORKSPACE_PATH = "code-workspace-code_gen-aa67a54b26fc4edd8bab69ddc8ba4818-2026-05-10_03-47-00"

# Causal bundle file(s).
# Accepts: a single path string, a list of paths, or None (falls back to inputs.json).
# When a list is given, triples from all files are merged and deduplicated by (head, relationship, tail).
CAUSAL_FILE_PATH = [
    "/Users/rapepong/Downloads/note1.txt-bundle-2026-05-10T04-42-38-442Z.json",
    "/Users/rapepong/Downloads/transcript_of_field_investigation.m4a-bundle-2026-05-10T04-42-32-468Z.json",
]

# --- Auth: pick ONE method ---
# Method A: API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Method B: Application Default Credentials (ADC)
# Run once in terminal:  gcloud auth application-default login
# Then set USE_ADC = True. Routes through Vertex AI -- set project + location.
USE_ADC = True
GCP_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT", "senior-cpe-04")
GCP_LOCATION = os.getenv("VERTEX_AI_LOCATION", "us-central1")

MODEL = "gemini-2.5-pro"

# Pass 2 threshold: entities with behavior_score <= this go to interface audit
PASS2_SCORE_THRESHOLD = 3

# Output directory for verdict JSONs
OUTPUT_DIR = "judge_output/newpipe2"

## 📦 Imports

In [166]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from google import genai
from google.genai.types import GenerateContentConfig
from IPython.display import display, Markdown

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("google-genai version:", genai.__version__)

google-genai version: 1.69.0


## 📂 Load Workspace Data

In [167]:
ws = Path(WORKSPACE_PATH)
artifacts = ws / "artifacts"
checkpoints = ws / "checkpoints"

# --- Entity dependency graph ---
deps_path = checkpoints / "state1c_entity_dependencies.json"
with open(deps_path) as f:
    deps_data = json.load(f)
dep_edges = deps_data["edges"]   # [{from, to, reason}]
entity_order = deps_data.get("order", [])
print(f"Dependency edges: {len(dep_edges)}")

# --- Metric contracts ---
with open(artifacts / "metric_contracts.json") as f:
    mc_data = json.load(f)
metrics = mc_data["metrics"]  # [{name, entities, required_attrs, rationale}]
print(f"Metric contracts: {len(metrics)}")

# --- Causal triples ---
def _extract_triples_from_chunks(chunks: list) -> list[dict]:
    triples = []
    for chunk in chunks:
        for cls in chunk.get("classes", []):
            for e in cls.get("extracted", []):
                triples.append(e)
    return triples

def _load_single(path: str) -> list[dict]:
    """Load triples from one causal file (bundle, flat list, or inputs.json dict)."""
    with open(path) as f:
        raw = json.load(f)
    if isinstance(raw, dict) and raw.get("export_type") == "causal_bundle":
        return _extract_triples_from_chunks(raw.get("raw_extraction", []))
    if isinstance(raw, list) and raw and "head" in raw[0]:
        return raw
    if isinstance(raw, dict) and "causalData" in raw:
        causal_str = raw["causalData"]
    else:
        raise ValueError(
            f"Unrecognised causal file format in {path}. "
            f"Keys: {list(raw.keys()) if isinstance(raw, dict) else type(raw)}"
        )
    if isinstance(causal_str, str):
        lines = causal_str.strip().split("\n", 1)
        json_str = lines[1] if len(lines) > 1 and lines[0].startswith("#") else causal_str
        chunks = json.loads(json_str)
    else:
        chunks = causal_str
    return _extract_triples_from_chunks(chunks)

def load_causal_triples(causal_file_path=None) -> list[dict]:
    """
    Load and merge causal triples.

    causal_file_path accepts:
      - None       : load from workspace/checkpoints/inputs.json
      - str        : load from single file
      - list[str]  : load and merge from all files, dedup by (head, relationship, tail)

    Each file supports three formats:
      1. Exported causal bundle  (export_type == "causal_bundle")
      2. Flat list               ([{head, relationship, tail, detail}])
      3. inputs.json dict        (causalData string key)
    """
    if causal_file_path is None:
        inputs_path = checkpoints / "inputs.json"
        with open(inputs_path) as f:
            inp = json.load(f)
        causal_str = inp.get("causalData", "")
        if isinstance(causal_str, str):
            lines = causal_str.strip().split("\n", 1)
            json_str = lines[1] if len(lines) > 1 and lines[0].startswith("#") else causal_str
            chunks = json.loads(json_str)
        else:
            chunks = causal_str
        return _extract_triples_from_chunks(chunks)

    paths = causal_file_path if isinstance(causal_file_path, list) else [causal_file_path]

    merged: list[dict] = []
    seen: set[tuple] = set()
    for p in paths:
        triples = _load_single(p)
        before = len(merged)
        for t in triples:
            key = (t.get("head", ""), t.get("relationship", ""), t.get("tail", ""))
            if key not in seen:
                seen.add(key)
                merged.append(t)
        added = len(merged) - before
        skipped = len(triples) - added
        print(f"  {p}: +{added} triples ({skipped} duplicates skipped)")

    return merged

causal_triples = load_causal_triples(CAUSAL_FILE_PATH)
print(f"Causal triples total: {len(causal_triples)}")

# --- Entity source files ---
entity_files = sorted((artifacts / "entities").glob("*.py"))
print(f"Entity files: {len(entity_files)}")

Dependency edges: 12
Metric contracts: 6
  /Users/rapepong/Downloads/note1.txt-bundle-2026-05-10T04-42-38-442Z.json: +48 triples (1 duplicates skipped)
  /Users/rapepong/Downloads/transcript_of_field_investigation.m4a-bundle-2026-05-10T04-42-32-468Z.json: +151 triples (5 duplicates skipped)
Causal triples total: 199
Entity files: 20


## 🗂️ Build Entity Index

In [168]:
NOISE_PREFIXES = {"entity", "canonical", "manual", "the"}

def entity_name_from_file(path: Path, label_map: dict[str, str] | None = None) -> str:
    eid = path.stem
    if label_map and eid in label_map:
        return label_map[eid].lower()
    parts = eid.split("-")
    name_parts = []
    for p in parts:
        if p in NOISE_PREFIXES:
            continue
        if p.isdigit():
            continue
        if len(p) > 10 and p.isalnum():
            continue
        name_parts.append(p)
    return " ".join(name_parts)

def fuzzy_match(contract_name: str, entity_name: str, threshold: int = 1) -> bool:
    stop = {"the","a","an","of","at","in","and","or","to","for","from","by","with","is","are","was","be","as"}
    def sig(s: str) -> set[str]:
        return set(s.lower().replace("-", " ").split()) - stop
    return len(sig(contract_name) & sig(entity_name)) >= threshold

def triple_match(triple: dict, entity_name: str) -> bool:
    head = triple.get("head", "").lower()
    tail = triple.get("tail", "").lower()
    en = entity_name.lower()
    en_words = set(en.split())
    stop = {"the","a","an","of","at","in","and","or","to","for"}
    for text in (head, tail):
        if en in text or text in en:
            return True
        text_words = set(text.replace("-", " ").split())
        overlap = (en_words - stop) & (text_words - stop)
        if len(overlap) >= 2:
            return True
        if len(en_words) == 1 and overlap:
            return True
    return False

# Load label map from inputs.json
label_map: dict[str, str] = {}
try:
    with open(checkpoints / "inputs.json") as f:
        inp = json.load(f)
    for entry in inp.get("userEntityList", []):
        label_map[entry["id"]] = entry["label"]
    print(f"Loaded {len(label_map)} entity labels from inputs.json")
except Exception as e:
    print(f"Could not load label map: {e}")

# Build per-entity context index
entity_index: dict[str, dict] = {}
for ef in entity_files:
    eid = ef.stem
    human_name = entity_name_from_file(ef, label_map)
    outgoing = [e for e in dep_edges if e["from"] == eid]
    incoming = [e for e in dep_edges if e["to"] == eid]
    required_attrs: list[dict] = []
    for m in metrics:
        matched = [me for me in m["entities"] if fuzzy_match(me, human_name)]
        if matched:
            required_attrs.append({
                "metric": m["name"],
                "label": m["label"],
                "matched_as": matched,
                "attrs": m.get("required_attrs", []),
                "rationale": m.get("rationale", ""),
            })
    relevant_triples = [t for t in causal_triples if triple_match(t, human_name)]
    entity_index[eid] = {
        "eid": eid,
        "human_name": human_name,
        "file": ef,
        "outgoing": outgoing,
        "incoming": incoming,
        "required_attrs": required_attrs,
        "causal_triples": relevant_triples,
    }

print(f"\nIndexed {len(entity_index)} entities:")
for eid, ctx in entity_index.items():
    print(f"  [{ctx['human_name']!r:35s}] out={len(ctx['outgoing'])} in={len(ctx['incoming'])} metrics={len(ctx['required_attrs'])} triples={len(ctx['causal_triples'])}")

Loaded 20 entity labels from inputs.json

Indexed 20 entities:
  ['compactor_bin'                    ] out=1 in=1 metrics=0 triples=0
  ['faculties'                        ] out=0 in=0 metrics=0 triples=1
  ['green society'                    ] out=3 in=1 metrics=1 triples=3
  ['n15'                              ] out=0 in=1 metrics=1 triples=1
  ['students'                         ] out=0 in=0 metrics=0 triples=6
  ['the lab'                          ] out=0 in=0 metrics=0 triples=1
  ['trash bins'                       ] out=0 in=0 metrics=0 triples=10
  ['garbage buffer'                   ] out=0 in=1 metrics=1 triples=5
  ['garbage truck'                    ] out=0 in=0 metrics=1 triples=3
  ['departments'                      ] out=1 in=0 metrics=0 triples=2
  ['waste'                            ] out=0 in=1 metrics=0 triples=26
  ['event'                            ] out=1 in=0 metrics=0 triples=2
  ['pest'                             ] out=1 in=0 metrics=0 triples=0
  ['problem'

## 🗃️ Policy Index

Loads policy outlines and source files. Builds:
- `entity_policies[eid]` — policies whose `target_entity_id == eid` (used in Pass 1 prompts)
- `find_policies_for_edge(edge)` — fuzzy-matches dep-graph edge → implementing policy files (used in Pass 2 prompts)

In [169]:
# --- Load policy outline (rule_id → target_entity_id + target_method) ---
with open(checkpoints / "state1b_policy_outline.json") as f:
    _po = json.load(f)
policy_outline: list[dict] = _po["policies"]
print(f"Policy outlines: {len(policy_outline)}")

# --- Load all policy source files ---
policy_files: dict[str, Path] = {p.stem: p for p in (artifacts / "policies").glob("*.py")}
print(f"Policy files found: {len(policy_files)}")

# --- entity_policies: eid → [(rule_id, code), ...] ---
entity_policies: dict[str, list[tuple[str, str]]] = {}
for p in policy_outline:
    target = p.get("target_entity_id", "")
    rule_id = p["rule_id"]
    if target and rule_id in policy_files:
        code = policy_files[rule_id].read_text()
        entity_policies.setdefault(target, []).append((rule_id, code))

print(f"\nEntities with associated policies: {len(entity_policies)}")
for eid, pols in entity_policies.items():
    print(f"  {eid}: {[r for r, _ in pols]}")

# --- find_policies_for_edge ---
_STOP = {"the","a","an","of","at","in","and","or","to","for","from","by","with","is","are","was"}

def find_policies_for_edge(edge: dict) -> list[tuple[str, str]]:
    """
    Fuzzy-match a dep-graph edge to its implementing policy files.

    score = word_overlap(edge.reason, policy.label) + 2 * (target_entity in {from, to})
    Returns [(rule_id, code), ...] sorted by score desc, up to 3 matches with score >= 3.
    """
    from_eid = edge["from"]
    to_eid   = edge["to"]
    reason_words = set(edge["reason"].lower().replace("-", " ").split()) - _STOP

    scored: list[tuple[int, str]] = []
    for p in policy_outline:
        rule_id = p["rule_id"]
        if rule_id not in policy_files:
            continue
        label_words = set(p.get("label", "").lower().replace("-", " ").split()) - _STOP
        overlap = len(reason_words & label_words)
        target_bonus = 2 if p.get("target_entity_id", "") in (from_eid, to_eid) else 0
        score = overlap + target_bonus
        if score >= 3:
            scored.append((score, rule_id))

    scored.sort(reverse=True)
    return [(rule_id, policy_files[rule_id].read_text()) for _, rule_id in scored[:3]]

# Sanity check
print("\nEdge → policy match preview:")
for edge in dep_edges[:5]:
    matched = find_policies_for_edge(edge)
    label = f"{edge['from']} → {edge['to']}"
    print(f"  {label[:60]:60s} → {[r for r, _ in matched]}")

Policy outlines: 32
Policy files found: 32

Entities with associated policies: 15
  waste_sorting_plant: ['send_food_waste_to_bma']
  entity-canonical-8-bma: ['dispose_of_received_waste']
  entity-89-departments: ['aggregate_departmental_recyclables', 'label_departmental_waste', 'schedule_large_item_pickup']
  junk_shop: ['collect_departmental_recyclables']
  entity-109-green-society: ['record_departmental_waste_collection', 'report_collection_for_payment', 'send_general_waste_to_n15', 'send_batteries_for_disposal', 'send_usable_ewaste_to_foundation', 'warn_about_overflow']
  environment: ['distribute_recycling_revenue', 'sell_scrap_metal']
  entity-13-students: ['dispose_in_infectious_waste_bin', 'dispose_in_hazardous_waste_bin', 'report_overflowing_bins']
  entity-100-faculties: ['accept_ewaste_donations']
  mirror_foundation: ['use_ewaste_for_repairs']
  entity-161-the-lab: ['dispose_of_lab_waste']
  waste_collector: ['compost_tree_scraps', 'collect_and_sort_indoor_waste', 'weigh_an

## 📝 Prompt Generator

In [170]:
PASS1_SYSTEM = """\
You are an expert simulation engineer auditing auto-generated Python entity classes.
Each entity is a component in a discrete-event waste management simulation.

## Architecture: Policy-as-Behaviour-Adapter

IMPORTANT: This codebase separates state from behaviour.
- Entities are stateful hook providers: they hold state, expose named methods,
  and implement on_query() / on_interact() interfaces.
- Policies are behaviour adapters: each Policy class implements one causal rule
  by looking up entities via env.get_entity(id) and calling their methods.

An entity that looks "hollow" by conventional standards may be CORRECT —
its job is to provide a clean, correct interface, not to drive its own behaviour.

Your job: verify that the entity provides a complete, correct interface so that
its associated policies can function without runtime errors.

## Scoring rubric for behavior_score (0-3)
- 3: Complete interface — all policy-callable methods present with correct signatures;
     on_query() returns all required metric attrs; state attributes well-initialised
- 2: Mostly present — minor attrs missing from on_query() or shallow state init;
     policy calls will work but data quality may suffer
- 1: Broken interface — key methods missing or wrong signatures; policies will fail at runtime
- 0: Hollow stub — no usable interface; policies cannot function

## Output format
Return ONLY valid JSON matching the schema provided.
"""

PASS1_SCHEMA = {
    "type": "object",
    "properties": {
        "entity": {"type": "string"},
        "behavior_score": {"type": "integer"},
        "verdict": {"type": "string", "enum": ["pass", "warn", "fail"]},
        "causal_claim_checks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "claim": {"type": "string"},
                    "status": {"type": "string", "enum": ["FOUND", "PARTIAL", "MISSING"]},
                    "evidence": {"type": "string"},
                },
                "required": ["claim", "status", "evidence"],
            },
        },
        "missing_attrs": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Metric attrs required by contracts but absent or wrong in on_query()",
        },
        "interface_concerns": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Method signature mismatches, wrong action strings, type issues that will break policy calls",
        },
        "summary": {"type": "string"},
    },
    "required": ["entity", "behavior_score", "verdict", "causal_claim_checks", "missing_attrs", "interface_concerns", "summary"],
}

def build_causal_claims(ctx: dict) -> str:
    lines = []
    idx = 1
    for e in ctx["outgoing"]:
        lines.append(f"{idx}. (OUTGOING) \"{ctx['human_name']} → {e['to'].replace('-',' ')}\": {e['reason']}")
        idx += 1
    for e in ctx["incoming"]:
        lines.append(f"{idx}. (INCOMING) \"{e['from'].replace('-',' ')} → {ctx['human_name']}\": {e['reason']}")
        idx += 1
    for t in ctx["causal_triples"][:10]:
        lines.append(f"{idx}. (CAUSAL TRIPLE) \"{t['head']}\" --[{t['relationship']}]--> \"{t['tail']}\" {t.get('detail', '')}")
        idx += 1
    return "\n".join(lines) if lines else "No causal claims found for this entity."

def build_metric_section(ctx: dict) -> str:
    if not ctx["required_attrs"]:
        return "No metric contracts reference this entity."
    lines = []
    for m in ctx["required_attrs"]:
        attr_list = ", ".join(a["attr"] for a in m["attrs"]) if m["attrs"] else "(none specified)"
        lines.append(f"- Metric `{m['metric']}` ({m['label']}): required attrs = [{attr_list}]")
        lines.append(f"  Rationale: {m['rationale']}")
    return "\n".join(lines)

def build_policy_section(eid: str, max_policies: int = 5) -> str:
    pols = entity_policies.get(eid, [])
    if not pols:
        return "No policies target this entity directly."
    lines = [f"### Policy: `{rule_id}`\n```python\n{code}\n```" for rule_id, code in pols[:max_policies]]
    if len(pols) > max_policies:
        lines.append(f"... and {len(pols) - max_policies} more policies (truncated)")
    return "\n\n".join(lines)

def build_pass1_prompt(ctx: dict) -> str:
    code = ctx["file"].read_text()
    return f"""{PASS1_SYSTEM}

## Entity: `{ctx['eid']}` (human name: {ctx['human_name']})

---
### Causal Claims (context — behaviour lives in policies, not entity code)
Use to understand WHAT the entity should support; do NOT judge whether it self-drives the behaviour.

{build_causal_claims(ctx)}

---
### Metric Attribute Contracts
`on_query()` must expose these keys for metrics to work.

{build_metric_section(ctx)}

---
### Associated Policies (these call methods on this entity)
Verify entity exposes all methods/attrs the policies below rely on.

{build_policy_section(ctx['eid'])}

---
### Entity Source Code

```python
{code}
```

---
Score based on interface completeness against the policies above, not on self-contained behaviour.
Return JSON only.
"""

# Preview one prompt
sample_eid = list(entity_index.keys())[0]
sample_prompt = build_pass1_prompt(entity_index[sample_eid])
print(f"Sample prompt for {sample_eid} — {len(sample_prompt)} chars")
print(sample_prompt[:1500], "\n...(truncated)")

Sample prompt for compactor_bin — 6593 chars
You are an expert simulation engineer auditing auto-generated Python entity classes.
Each entity is a component in a discrete-event waste management simulation.

## Architecture: Policy-as-Behaviour-Adapter

IMPORTANT: This codebase separates state from behaviour.
- Entities are stateful hook providers: they hold state, expose named methods,
  and implement on_query() / on_interact() interfaces.
- Policies are behaviour adapters: each Policy class implements one causal rule
  by looking up entities via env.get_entity(id) and calling their methods.

An entity that looks "hollow" by conventional standards may be CORRECT —
its job is to provide a clean, correct interface, not to drive its own behaviour.

Your job: verify that the entity provides a complete, correct interface so that
its associated policies can function without runtime errors.

## Scoring rubric for behavior_score (0-3)
- 3: Complete interface — all policy-callable methods prese

## 🤖 Gemini Client Setup

In [171]:
import time

if USE_ADC:
    if not GCP_PROJECT:
        raise ValueError("Set GCP_PROJECT when using ADC (Vertex AI requires a project)")
    client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
    print(f"Auth: ADC (Vertex AI) — project={GCP_PROJECT} location={GCP_LOCATION}")
else:
    if not GEMINI_API_KEY:
        raise ValueError("Set GEMINI_API_KEY or enable USE_ADC in the Config cell")
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Auth: API key")

_RETRY_DELAYS = [30, 60, 120, 240]

def _is_429(exc: Exception) -> bool:
    msg = str(exc).lower()
    typ = type(exc).__name__
    return (
        "429" in msg
        or "resourceexhausted" in typ.lower()
        or "resource_exhausted" in msg
        or "quota" in msg
        or ("rate" in msg and "limit" in msg)
    )

def call_gemini(prompt: str, schema: dict) -> dict:
    """Gemini call with up to 4 retries on 429; falls back to error stub on exhaustion."""
    last_exc: Exception | None = None
    for attempt, delay in enumerate([0] + _RETRY_DELAYS, start=1):
        if delay:
            print(f"    [retry {attempt-1}/4] 429 received — waiting {delay}s...", flush=True)
            time.sleep(delay)
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0.1,
                ),
            )
            return json.loads(response.text)
        except Exception as exc:
            if _is_429(exc):
                last_exc = exc
                if attempt <= 4:
                    continue
                break
            raise

    print(f"    [fallback] 429 unresolved after 4 retries: {last_exc}")
    return {
        "entity": "unknown",
        "behavior_score": -1,
        "verdict": "error",
        "causal_claim_checks": [],
        "missing_attrs": [],
        "interface_concerns": [],
        "summary": f"Skipped — quota exhausted after 4 retries: {last_exc}",
    }

print(f"Gemini client ready — model={MODEL}")

Auth: ADC (Vertex AI) — project=senior-cpe-04 location=us-central1
Gemini client ready — model=gemini-2.5-pro


## 🔍 Pass 1 — Per-Entity Interface Audit

In [172]:
pass1_results: dict[str, dict] = {}
pass1_errors: dict[str, str] = {}

eids = list(entity_index.keys())
print(f"Running Pass 1 on {len(eids)} entities...\n")

for i, eid in enumerate(eids, 1):
    ctx = entity_index[eid]
    print(f"[{i}/{len(eids)}] {eid} ... ", end="", flush=True)
    try:
        prompt = build_pass1_prompt(ctx)
        result = call_gemini(prompt, PASS1_SCHEMA)
        pass1_results[eid] = result

        out_path = Path(OUTPUT_DIR) / f"pass1_{eid}.json"
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)

        score = result.get("behavior_score", "?")
        verdict = result.get("verdict", "?")
        print(f"{verdict.upper()} (score={score})")
    except Exception as exc:
        pass1_errors[eid] = str(exc)
        print(f"ERROR: {exc}")

    if i < len(eids):
        time.sleep(1.5)

print(f"\nDone. {len(pass1_results)} succeeded, {len(pass1_errors)} errors.")

Running Pass 1 on 20 entities...

[1/20] compactor_bin ... PASS (score=3)
[2/20] entity-100-faculties ... PASS (score=3)
[3/20] entity-109-green-society ... FAIL (score=1)
[4/20] entity-127-n15 ... PASS (score=3)
[5/20] entity-13-students ... PASS (score=3)
[6/20] entity-161-the-lab ... PASS (score=3)
[7/20] entity-171-trash-bins ... PASS (score=3)
[8/20] entity-18-garbage-buffer ... PASS (score=3)
[9/20] entity-38-garbage-truck ... WARN (score=2)
[10/20] entity-89-departments ... PASS (score=3)
[11/20] entity-canonical-0-waste ... PASS (score=3)
[12/20] entity-canonical-13-event ... FAIL (score=1)
[13/20] entity-canonical-16-pest ... PASS (score=3)
[14/20] entity-canonical-4-problem ... PASS (score=3)
[15/20] entity-canonical-8-bma ... FAIL (score=1)
[16/20] junk_shop ... PASS (score=3)
[17/20] mirror_foundation ... PASS (score=3)
[18/20] sorter ... PASS (score=3)
[19/20] waste_collector ... FAIL (score=1)
[20/20] waste_sorting_plant ... PASS (score=3)

Done. 20 succeeded, 0 errors.


## 💾 Reload Pass 1 Results

Run this cell instead of the Pass 1 loop after a kernel restart.

In [173]:
pass1_results: dict[str, dict] = {}
pass1_errors: dict[str, str] = {}

for f in sorted(Path(OUTPUT_DIR).glob("pass1_*.json")):
    eid = f.stem.removeprefix("pass1_")
    with open(f) as fh:
        pass1_results[eid] = json.load(fh)

print(f"Reloaded {len(pass1_results)} Pass 1 results from {OUTPUT_DIR}/")
for eid, r in pass1_results.items():
    print(f"  {eid}: score={r.get('behavior_score','?')} verdict={r.get('verdict','?')}")

Reloaded 20 Pass 1 results from judge_output/newpipe2/
  compactor_bin: score=3 verdict=pass
  entity-100-faculties: score=3 verdict=pass
  entity-109-green-society: score=1 verdict=fail
  entity-127-n15: score=3 verdict=pass
  entity-13-students: score=3 verdict=pass
  entity-161-the-lab: score=3 verdict=pass
  entity-171-trash-bins: score=3 verdict=pass
  entity-18-garbage-buffer: score=3 verdict=pass
  entity-38-garbage-truck: score=2 verdict=warn
  entity-89-departments: score=3 verdict=pass
  entity-canonical-0-waste: score=3 verdict=pass
  entity-canonical-13-event: score=1 verdict=fail
  entity-canonical-16-pest: score=3 verdict=pass
  entity-canonical-4-problem: score=3 verdict=pass
  entity-canonical-8-bma: score=1 verdict=fail
  junk_shop: score=3 verdict=pass
  mirror_foundation: score=3 verdict=pass
  sorter: score=3 verdict=pass
  waste_collector: score=1 verdict=fail
  waste_sorting_plant: score=3 verdict=pass


## 📊 Pass 1 Results

In [174]:
def verdict_emoji(v: str) -> str:
    return {"pass": "✅", "warn": "⚠️", "fail": "❌"}.get(v, "❓")

rows = []
for eid, r in pass1_results.items():
    claims = r.get("causal_claim_checks", [])
    found   = sum(1 for c in claims if c["status"] == "FOUND")
    partial = sum(1 for c in claims if c["status"] == "PARTIAL")
    missing = sum(1 for c in claims if c["status"] == "MISSING")
    rows.append({
        "entity": eid,
        "verdict": f"{verdict_emoji(r.get('verdict','?'))} {r.get('verdict','?')}",
        "score": r.get("behavior_score", "?"),
        "claims (F/P/M)": f"{found}/{partial}/{missing}",
        "missing_attrs": len(r.get("missing_attrs", [])),
        "interface_concerns": len(r.get("interface_concerns", [])),
        "summary": r.get("summary", "")[:120],
    })

df = pd.DataFrame(rows).sort_values("score")
pd.set_option("display.max_colwidth", 130)
display(df.reset_index(drop=True))

if pass1_errors:
    print("\nErrors:")
    for eid, err in pass1_errors.items():
        print(f"  {eid}: {err}")

,entity,verdict,score,claims (F/P/M),missing_attrs,interface_concerns,summary
0,entity-109-green-society,❌ fail,1,3/0/1,1,2,"The entity correctly implements most methods required by its associated policies for directing waste streams. However, t"
1,entity-canonical-8-bma,❌ fail,1,1/1/4,0,2,"The entity correctly implements the waste disposal pathway, satisfying the `dispose_of_received_waste` policy and the `w"
2,waste_collector,❌ fail,1,3/1/0,0,1,"The entity correctly implements most methods required by its associated policies, such as for indoor/outdoor collection"
3,entity-canonical-13-event,❌ fail,1,2/1/0,1,1,The entity has a critical interface failure in its `on_query` method that will cause the `schedule_event_waste_pickup` p
4,entity-38-garbage-truck,⚠️ warn,2,3/0/0,2,0,The entity provides a robust interface for policies with a well-defined state machine and methods like `start_collection
5,compactor_bin,✅ pass,3,1/0/1,0,0,The CompactorBin entity is a well-designed passive component. It correctly implements the `on_interact` method to allow
6,sorter,✅ pass,3,2/0/0,0,0,The Sorter entity provides a complete and correct interface for its associated policies. It correctly implements the `so
7,mirror_foundation,✅ pass,3,1/0/0,0,0,The entity provides a complete and correct interface. It implements the `use_donations_for_repairs` method with the corr
8,junk_shop,✅ pass,3,0/0/1,0,0,The entity provides a complete and correct interface for the associated `collect_departmental_recyclables` policy. It co
9,entity-canonical-4-problem,✅ pass,3,2/0/0,0,0,The entity provides a complete and correct interface for its associated policies. It correctly implements the `__init__`


## 🔗 Pass 2 — Policy-Entity Contract Audit

Flags edges where at least one endpoint has `behavior_score <= PASS2_SCORE_THRESHOLD` or `verdict != pass`.
For each flagged edge, loads the implementing policy and audits the three-way contract:
**policy ↔ FROM entity ↔ TO entity**.

In [175]:
# ── Flag entities for Pass 2 ────────────────────────────────────────────────
def _iter_pass1(results):
    if isinstance(results, pd.DataFrame):
        for _, row in results.iterrows():
            eid = row.get("entity", "")
            try:
                score = int(row.get("score", 3))
            except (ValueError, TypeError):
                score = 3
            raw_verdict = str(row.get("verdict", "pass"))
            verdict = raw_verdict.split()[-1] if raw_verdict else "pass"
            yield eid, score, verdict
    else:
        for eid, r in results.items():
            yield eid, r.get("behavior_score", 3), r.get("verdict", "pass")

flagged_eids = {
    eid for eid, score, verdict in _iter_pass1(pass1_results)
    if score <= PASS2_SCORE_THRESHOLD or verdict != "pass"
}
print(f"Flagged entities (score <= {PASS2_SCORE_THRESHOLD} OR verdict != pass): {len(flagged_eids)}")

flagged_edges = [e for e in dep_edges if e["from"] in flagged_eids or e["to"] in flagged_eids]
seen_pairs: set = set()
unique_flagged_edges: list[dict] = []
for e in flagged_edges:
    pair = (e["from"], e["to"])
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_flagged_edges.append(e)

print(f"Flagged edges for Pass 2: {len(unique_flagged_edges)}")
for e in unique_flagged_edges:
    print(f"  {e['from']} → {e['to']}: {e['reason']}")

# ── Pass 2 Schema ─────────────────────────────────────────────────────────────
PASS2_SCHEMA = {
    "type": "object",
    "properties": {
        "edge": {"type": "string"},
        "compatible": {"type": "boolean"},
        "issues": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Specific contract mismatches: wrong method name, wrong entity ID in get_entity(), missing attr key, type conflict, etc.",
        },
        "fix_suggestions": {"type": "array", "items": {"type": "string"}},
        "summary": {"type": "string"},
    },
    "required": ["edge", "compatible", "issues", "fix_suggestions", "summary"],
}

def _get_pass1_concerns(results, eid: str) -> str:
    if isinstance(results, pd.DataFrame):
        rows = results[results["entity"] == eid]
        if rows.empty:
            return "None flagged."
        val = rows.iloc[0].get("interface_concerns", "")
        return str(val) if val else "None flagged."
    else:
        r = results.get(eid, {})
        concerns = r.get("interface_concerns", [])
        return "\n".join(concerns) if concerns else "None flagged."

def build_pass2_prompt(edge: dict) -> str:
    from_eid = edge["from"]
    to_eid   = edge["to"]
    reason   = edge["reason"]

    from_code = entity_index[from_eid]["file"].read_text() if from_eid in entity_index else "(file not found)"
    to_code   = entity_index[to_eid]["file"].read_text()   if to_eid   in entity_index else "(file not found)"

    from_concerns = _get_pass1_concerns(pass1_results, from_eid)
    to_concerns   = _get_pass1_concerns(pass1_results, to_eid)

    edge_policies = find_policies_for_edge(edge)
    if edge_policies:
        policy_block = "\n\n".join(
            f"### Policy: `{rule_id}`\n```python\n{code}\n```"
            for rule_id, code in edge_policies
        )
    else:
        policy_block = (
            "No policy matched for this edge.\n"
            "Check manually: behaviour may be in environment.py or in entity step()."
        )

    return f"""\
You are auditing the three-way contract between two entity classes and the policy that connects them.

## Architecture note
In this simulation, entities do NOT implement causal behaviour directly.
Policies are behaviour adapters: they call entity methods via env.get_entity(id).
Verify that the policy's calls to each entity are correct.

## Edge under review
FROM: `{from_eid}` → TO: `{to_eid}`
Causal reason: "{reason}"

## Prior audit concerns
FROM entity concerns: {from_concerns}
TO entity concerns: {to_concerns}

## Check points
1. Does the policy call env.get_entity() with the correct entity IDs (match actual file stems)?
2. Does the policy call the correct method names on the FROM entity?
3. Does the policy call the correct method names on the TO entity?
4. Are attribute names in context / on_query() keys consistent between policy and entity?
5. Are there type or unit mismatches (e.g. policy passes kg, entity expects count)?
6. Does the FROM entity expose what the policy reads from it?
7. Does the TO entity's receiving method handle the payload the policy sends?

## Implementing Policy/Policies

{policy_block}

## FROM entity source: `{from_eid}`

```python
{from_code}
```

## TO entity source: `{to_eid}`

```python
{to_code}
```

Return JSON only.
"""

# ── Run Pass 2 ────────────────────────────────────────────────────────────────
pass2_results: dict[str, dict] = {}
pass2_errors: dict[str, str] = {}

print(f"\nRunning Pass 2 on {len(unique_flagged_edges)} edges...\n")

for i, edge in enumerate(unique_flagged_edges, 1):
    pair_key = f"{edge['from']} → {edge['to']}"
    matched_policies = [r for r, _ in find_policies_for_edge(edge)]
    print(f"[{i}/{len(unique_flagged_edges)}] {pair_key}")
    print(f"    policies matched: {matched_policies}")
    print("    ... ", end="", flush=True)

    if edge["from"] not in entity_index or edge["to"] not in entity_index:
        print("SKIP (entity file missing)")
        continue

    try:
        prompt = build_pass2_prompt(edge)
        result = call_gemini(prompt, PASS2_SCHEMA)
        pass2_results[pair_key] = result

        out_path = Path(OUTPUT_DIR) / f"pass2_{edge['from']}__{edge['to']}.json"
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)

        compat = "✅ compatible" if result.get("compatible") else f"❌ {len(result.get('issues', []))} issues"
        print(compat)
    except Exception as exc:
        pass2_errors[pair_key] = str(exc)
        print(f"ERROR: {exc}")

    if i < len(unique_flagged_edges):
        time.sleep(1.5)

print(f"\nDone. {len(pass2_results)} succeeded, {len(pass2_errors)} errors.")

Flagged entities (score <= 3 OR verdict != pass): 20
Flagged edges for Pass 2: 12
  waste_sorting_plant → entity-canonical-8-bma: The waste sorting plant sends food waste to the BMA for disposal, as stated in the 'send_food_waste_to_bma' policy.
  entity-109-green-society → junk_shop: The waste manager from Green Society accompanies the junk shop to record waste collection from departments, as described in the 'record_departmental_waste_collection' policy and causal data.
  entity-109-green-society → entity-127-n15: Green society sends cleaned general waste to N15 to be used as fuel, according to the 'send_general_waste_to_n15' policy.
  entity-109-green-society → mirror_foundation: Green society sends usable electronic waste to the Mirror Foundation for reuse, as per the 'send_usable_ewaste_to_foundation' policy.
  waste_collector → entity-18-garbage-buffer: Waste collectors transfer collected waste to a garbage buffer (waste holding cage) before transport to the main plant, as per th

## 📊 Pass 2 Results

In [176]:
rows2 = []
for pair_key, r in pass2_results.items():
    rows2.append({
        "edge": pair_key,
        "compatible": "✅" if r.get("compatible") else "❌",
        "issues": len(r.get("issues", [])),
        "fixes": len(r.get("fix_suggestions", [])),
        "summary": r.get("summary", "")[:140],
    })

if rows2:
    df2 = pd.DataFrame(rows2)
    display(df2)
else:
    print("No Pass 2 results (no flagged edges or all skipped).")

for pair_key, r in pass2_results.items():
    if r.get("compatible"):
        continue
    print(f"\n{'='*70}")
    print(f"Edge: {pair_key}")
    print(f"Summary: {r.get('summary','')}")
    for issue in r.get("issues", []):
        print(f"  ❌ {issue}")
    for fix in r.get("fix_suggestions", []):
        print(f"  💡 {fix}")

,edge,compatible,issues,fixes,summary
0,waste_sorting_plant → entity-canonical-8-bma,✅,0,0,The contract is compatible. The 'send_food_waste_to_bma' policy correctly calls the 'send_waste_for_disposal' method on the 'w...
1,entity-109-green-society → junk_shop,❌,4,2,"The contract is incompatible due to a critical data source mismatch. The `record_departmental_waste_collection` policy, which ..."
2,entity-109-green-society → entity-127-n15,✅,0,0,The contract is compatible. The 'send_general_waste_to_n15' policy correctly calls the 'send_waste_for_fuel' method on the Gre...
3,entity-109-green-society → mirror_foundation,✅,0,0,The contract is compatible. The `send_usable_ewaste_to_foundation` policy correctly calls the `send_for_reuse` method on the `...
4,waste_collector → entity-18-garbage-buffer,✅,0,0,The 'weigh_and_transfer_to_buffer' policy correctly orchestrates the interaction. The policy identifies the correct FROM ('was...
5,waste_collector → waste_sorting_plant,❌,3,1,"Incompatible. The policy incorrectly attempts to get the collector's held waste via `on_query('held_waste')`, which is not imp..."
6,sorter → waste_sorting_plant,❌,3,3,"The contract is incompatible. The `waste_sorting_plant` completely ignores the `sorter` entity and its associated policy, impl..."
7,sorter → compactor_bin,✅,0,0,The contract is compatible. The 'compact_unsortable_waste' policy correctly calls the 'load_into_compactor' method on the 'sor...
8,entity-89-departments → entity-109-green-society,❌,4,3,"The implementation is incompatible with the causal description. The policy triggers the department to make a request, but the ..."
9,entity-canonical-13-event → entity-canonical-8-bma,❌,3,3,The contract is incompatible. The policy cannot correctly retrieve waste data from the FROM entity because it uses the wrong m...



Edge: entity-109-green-society → junk_shop
Summary: The contract is incompatible due to a critical data source mismatch. The `record_departmental_waste_collection` policy, which governs the Green Society's recording action, targets `Entity_Departments`. However, the `junk_shop` it is supposed to be observing collects from `Entity_Faculties`. This discrepancy means the Green Society will fail to record the junk shop's actual activities, violating the causal relationship. There is also an inconsistent access pattern for retrieving waste data between the policy and the entity.
  ❌ Data Source Mismatch: The `record_departmental_waste_collection` policy iterates over `env.get_entities_by_type("Entity_Departments")` to find waste to record.
  ❌ Data Source Mismatch: The `junk_shop`'s `collect_from_buildings` method iterates over `env.get_entities_by_type("Entity_Faculties")` to collect waste.
  ❌ Contract Violation: The causal reason states the Green Society accompanies the junk shop to rec

## 📋 Final Report

In [177]:
def _to_eval_df(results) -> pd.DataFrame:
    if isinstance(results, pd.DataFrame):
        df = results.copy()
        df["verdict_clean"] = df["verdict"].astype(str).str.split().str[-1]
        df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(3).astype(int)
        return df
    else:
        rows = []
        for eid, r in results.items():
            claims = r.get("causal_claim_checks", [])
            found   = sum(1 for c in claims if c["status"] == "FOUND")
            partial = sum(1 for c in claims if c["status"] == "PARTIAL")
            missing = sum(1 for c in claims if c["status"] == "MISSING")
            rows.append({
                "entity": eid,
                "score": r.get("behavior_score", 3),
                "verdict": r.get("verdict", "pass"),
                "verdict_clean": r.get("verdict", "pass"),
                "claims (F/P/M)": f"{found}/{partial}/{missing}",
                "missing_attrs": len(r.get("missing_attrs", [])),
                "interface_concerns": len(r.get("interface_concerns", [])),
                "summary": r.get("summary", ""),
            })
        return pd.DataFrame(rows)

df_eval = _to_eval_df(pass1_results)

display(Markdown("""
### Architecture
Entities = stateful hook providers. Policies = behaviour adapters that call entity methods.
**Pass 1** judges entity interface completeness against its associated policies.
**Pass 2** audits the three-way contract: policy ↔ FROM entity ↔ TO entity.

### Scoring rubric (behavior_score)

| Score | Meaning |
|-------|---------|
| 3 | Complete interface — all policy-callable methods + on_query() attrs + well-initialised state |
| 2 | Mostly present — minor gaps; policy calls work but data quality may suffer |
| 1 | Broken interface — key methods missing; policies will fail at runtime |
| 0 | Hollow stub — no usable interface |

**Flagging rule for Pass 2:** score ≤ threshold **OR** verdict ≠ pass
"""))

total     = len(df_eval)
n_pass    = (df_eval["verdict_clean"] == "pass").sum()
n_warn    = (df_eval["verdict_clean"] == "warn").sum()
n_fail    = (df_eval["verdict_clean"] == "fail").sum()
avg_score = df_eval["score"].mean()

display(Markdown(f"""
### Aggregate Results — Pass 1 ({total} entities)

| | Count | % |
|---|---|---|
| ✅ Pass | {n_pass} | {n_pass/total*100:.0f}% |
| ⚠️ Warn | {n_warn} | {n_warn/total*100:.0f}% |
| ❌ Fail | {n_fail} | {n_fail/total*100:.0f}% |
| **Average score** | **{avg_score:.2f} / 3** | |
"""))

display(Markdown("### Per-Entity Verdicts"))
show_cols = ["entity", "score", "verdict", "claims (F/P/M)", "missing_attrs", "interface_concerns", "summary"]
available = [c for c in show_cols if c in df_eval.columns]
pd.set_option("display.max_colwidth", 100)
display(df_eval[available].sort_values("score").reset_index(drop=True))

display(Markdown("### Score Distribution"))
dist = df_eval.groupby("score")["entity"].count().reset_index()
dist.columns = ["score", "entity_count"]
dist["verdict_label"] = dist["score"].map({0: "hollow", 1: "broken", 2: "partial", 3: "complete"})
display(dist)

if "claims (F/P/M)" in df_eval.columns:
    def _parse_fpm(s):
        try:
            f, p, m = str(s).split("/")
            return int(f), int(p), int(m)
        except Exception:
            return 0, 0, 0
    df_eval[["_f","_p","_m"]] = df_eval["claims (F/P/M)"].apply(lambda s: pd.Series(_parse_fpm(s)))
    tf, tp, tm = df_eval["_f"].sum(), df_eval["_p"].sum(), df_eval["_m"].sum()
    tc = tf + tp + tm
    display(Markdown(f"""
### Causal Claim Coverage

| Status | Count | % |
|--------|-------|---|
| FOUND   | {tf} | {tf/tc*100:.1f}% |
| PARTIAL | {tp} | {tp/tc*100:.1f}% |
| MISSING | {tm} | {tm/tc*100:.1f}% |
| **Total** | **{tc}** | |
"""))

if 'pass2_results' in dir() and pass2_results:
    n_edges = len(pass2_results)
    n_incompat = sum(1 for r in pass2_results.values() if not r.get("compatible", True))
    display(Markdown(f"""
### Pass 2 — Policy-Entity Contract ({n_edges} edges audited)

| | Count |
|---|---|
| ✅ Compatible | {n_edges - n_incompat} |
| ❌ Incompatible | {n_incompat} |
"""))
else:
    display(Markdown("### Pass 2 — Not yet run or no flagged edges"))


### Architecture
Entities = stateful hook providers. Policies = behaviour adapters that call entity methods.
**Pass 1** judges entity interface completeness against its associated policies.
**Pass 2** audits the three-way contract: policy ↔ FROM entity ↔ TO entity.

### Scoring rubric (behavior_score)

| Score | Meaning |
|-------|---------|
| 3 | Complete interface — all policy-callable methods + on_query() attrs + well-initialised state |
| 2 | Mostly present — minor gaps; policy calls work but data quality may suffer |
| 1 | Broken interface — key methods missing; policies will fail at runtime |
| 0 | Hollow stub — no usable interface |

**Flagging rule for Pass 2:** score ≤ threshold **OR** verdict ≠ pass



### Aggregate Results — Pass 1 (20 entities)

| | Count | % |
|---|---|---|
| ✅ Pass | 15 | 75% |
| ⚠️ Warn | 1 | 5% |
| ❌ Fail | 4 | 20% |
| **Average score** | **2.55 / 3** | |


### Per-Entity Verdicts

,entity,score,verdict,claims (F/P/M),missing_attrs,interface_concerns,summary
0,entity-109-green-society,1,fail,3/0/1,1,2,The entity correctly implements most methods required by its associated policies for directing w...
1,entity-canonical-8-bma,1,fail,1/1/4,0,2,"The entity correctly implements the waste disposal pathway, satisfying the `dispose_of_received_..."
2,waste_collector,1,fail,3/1/0,0,1,"The entity correctly implements most methods required by its associated policies, such as for in..."
3,entity-canonical-13-event,1,fail,2/1/0,1,1,The entity has a critical interface failure in its `on_query` method that will cause the `schedu...
4,entity-38-garbage-truck,2,warn,3/0/0,2,0,The entity provides a robust interface for policies with a well-defined state machine and method...
5,compactor_bin,3,pass,1/0/1,0,0,The CompactorBin entity is a well-designed passive component. It correctly implements the `on_in...
6,sorter,3,pass,2/0/0,0,0,The Sorter entity provides a complete and correct interface for its associated policies. It corr...
7,mirror_foundation,3,pass,1/0/0,0,0,The entity provides a complete and correct interface. It implements the `use_donations_for_repai...
8,junk_shop,3,pass,0/0/1,0,0,The entity provides a complete and correct interface for the associated `collect_departmental_re...
9,entity-canonical-4-problem,3,pass,2/0/0,0,0,The entity provides a complete and correct interface for its associated policies. It correctly i...


### Score Distribution

,score,entity_count,verdict_label
0,1,4,broken
1,2,1,partial
2,3,15,complete



### Causal Claim Coverage

| Status | Count | % |
|--------|-------|---|
| FOUND   | 57 | 79.2% |
| PARTIAL | 5 | 6.9% |
| MISSING | 10 | 13.9% |
| **Total** | **72** | |



### Pass 2 — Policy-Entity Contract (12 edges audited)

| | Count |
|---|---|
| ✅ Compatible | 5 |
| ❌ Incompatible | 7 |
